In [20]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.optimize import minimize

from rich import print
import seaborn as sns
sns.set()

import ewatercycle
import ewatercycle.models
import ewatercycle.forcing
from scipy.stats import qmc
from ipywidgets import IntProgress

In [21]:
calibration_start_date = "1975-01-01"
calibration_end_date = "2010-12-31"

In [22]:
data = pd.read_csv("mohembo_daily_water_discharge_data.csv", index_col='date', parse_dates=True, dayfirst=True)
data_daily = data.resample('D').interpolate()
data_daily.columns = ['Discharge (m^3/s)']
data_daily = data_daily[~data_daily.index.year.isin([1974, 2021])]
data_daily = data_daily[(data_daily.index >= calibration_start_date) & (data_daily.index <= calibration_end_date)]
                         
forcing_path_ERA5 = Path.home() / "BEP-beau/BEP/code" / "CatchmentArea" / "ERA5" / "own_shapefile"
load_location = forcing_path_ERA5 / "work" / "diagnostic" / "script"  
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)

In [23]:
def RMSE(output, observed, start, end):
     # Making sure the given dates are in the correct format
    start = pd.to_datetime(start).tz_localize(None)
    end = pd.to_datetime(end).tz_localize(None)
    
    # Making sure the date of the modelled output and onserved input are in the correct format
    output.index = pd.to_datetime(output.index)
    observed.index = pd.to_datetime(observed.index)

    # Combine the model output and the observation into one data frame 
    hydro_data = pd.concat([output.reindex(observed.index, method='ffill'), observed], axis=1, keys=['model', 'observation'])
    hydro_data = hydro_data.dropna()
    
    # Making sure to only take the calibration period
    hydro_data = hydro_data[(hydro_data.index > start) & (hydro_data.index < end)]
    
    # Calculate the absolute square difference
    squarediff = (hydro_data['model'] - hydro_data['observation']) ** 2
    rootMeanSquareDiff = np.sqrt(np.mean(squarediff))
    
    return rootMeanSquareDiff

In [31]:
N = 250
s_0 = np.array([0,  100,  0,  5,  0])

# Define parameters and their corresponding boundary values 
param_names = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks", "FM"]
param_mins = np.array([0, 0.2, 500, 1.5, 0.01, 15, 0.0005, 0.00005, 0.00001])
param_maxs = np.array([8, 1.5, 2500, 4, 0.3, 90, 0.01, 0.002, 1])

#Fill the parameters array with N random values between each minimum and maximum 
sampler = qmc.LatinHypercube(d=len(param_names))
sample = sampler.random(n=N)
parameters = qmc.scale(sample, param_mins, param_maxs)

In [32]:
s_0 = np.array([0,  100,  0,  5, 0])

In [33]:
ensemble = []

for counter in range(N): 
    ensemble.append(ewatercycle.models.HBVLocal(forcing=ERA5_forcing))
    config_file, _ = ensemble[counter].setup(parameters = parameters[counter],  initial_storage=s_0)
    ensemble[counter].initialize(config_file)

In [34]:
f = IntProgress(min=0, max=N)
display(f)

objectives_RMSE = []
for ensembleMember in ensemble:
    Q_m = []
    time = []
    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        discharge_this_timestep = ensembleMember.get_value("Q")
        Q_m.append(discharge_this_timestep[0])
        time.append(ensembleMember.time_as_datetime)

    discharge_dataframe = pd.DataFrame({'model output': Q_m}, index=pd.to_datetime(time))
    fit_RMSE = RMSE(discharge_dataframe['model output'], data_daily['Discharge (m^3/s)'], calibration_start_time, calibration_end_time)
    objectives_RMSE.append(fit_RMSE)
    del Q_m, time, discharge_dataframe, fit_RMSE
    f.value += 1
    
for ensembleMember in ensemble:
    ensembleMember.finalize()

IntProgress(value=0, max=250)

In [37]:
parameters_RMSE_index = np.argmin(np.array(objectives_RMSE))
parameters_RMSE = parameters[parameters_RMSE_index]
parameters_RMSE

array([1.15381635e+00, 1.44109292e+00, 1.09051138e+03, 1.52100392e+00,
       2.11993707e-01, 8.08588873e+01, 8.81657547e-03, 5.98875895e-04,
       3.90705067e-01])